# NIfTI Medical Imaging with CLARYON

Demonstrates NIfTI loading, mask application, and 3D CNN training on synthetic volumes.

Since real NIfTI data requires nibabel and actual medical images, this notebook creates synthetic volumes to show the pipeline.

In [ ]:
import tempfile
from pathlib import Path

import numpy as np

# Create synthetic NIfTI-like data directory structure
tmp = tempfile.mkdtemp()
data_dir = Path(tmp) / "nifti_data" / "Train"
data_dir.mkdir(parents=True)

try:
    import nibabel as nib
    HAS_NIBABEL = True
except ImportError:
    HAS_NIBABEL = False
    print("nibabel not installed — showing config and pipeline structure only")

if HAS_NIBABEL:
    # Create 10 synthetic volumes (16x16x16) with labels
    for i in range(10):
        label = i % 2
        vol = np.random.randn(16, 16, 16).astype(np.float32)
        if label == 1:
            vol[4:12, 4:12, 4:12] += 2.0  # Signal in center for class 1
        mask = np.zeros((16, 16, 16), dtype=np.float32)
        mask[2:14, 2:14, 2:14] = 1.0

        nib.save(nib.Nifti1Image(vol, np.eye(4)), str(data_dir / f"subj_{i:03d}_image_{label}.nii.gz"))
        nib.save(nib.Nifti1Image(mask, np.eye(4)), str(data_dir / f"subj_{i:03d}_mask_{label}.nii.gz"))

    print(f"Created {len(list(data_dir.glob('*image*')))} image volumes")
    print(f"Created {len(list(data_dir.glob('*mask*')))} mask volumes")

In [ ]:
# Show how to configure NIfTI in YAML
config_example = """
data:
  imaging:
    path: data/nifti_dataset
    format: nifti
    image_pattern: "*image*"    # NEW: configurable image pattern (default: "*")
    mask_pattern: "*mask*"

models:
  - name: cnn_3d
    type: imaging
    params:
      epochs: 10
      batch_size: 4
"""
print(config_example)

In [ ]:
if HAS_NIBABEL:
    from claryon.io.nifti import load_nifti_dataset

    result = load_nifti_dataset(
        root=str(data_dir.parent),
        pet_pattern="*image*.nii*",
        mask_pattern="*mask*.nii*",
    )
    ds = result.get("train") or result.get("all")
    print(f"Loaded dataset: {ds.n_samples} samples, feature dim: {ds.n_features}")
    print(f"Labels: {np.unique(ds.y, return_counts=True)}")